# Chapter 21
## Gap Junctions
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter21.ipynb)

## About this chapter

Gap junctions are electrical synapses: each cell receives a current
proportional to the voltage difference from its neighbors,
$I_{{\rm gap},i}=g_{\rm gap}\sum_{j\ne i}(v_j-v_i)$, rather than a fixed
chemical reversal potential. `LIF_NETWORK_WITH_GJ` contrasts pure diffusive
coupling with an added spike-triggered voltage kick, using a discontinuous
LIF reset; `RESET_THRESHOLD` is a continuous WB reference trace used to
pick the LIF's reset/threshold pair. `WB_NETWORK_WITH_GJ` couples two
spiking WB neurons; `WB_NETWORK_WITH_GJ_SUBTHRESHOLD` isolates the
electrical-equalization effect before either cell spikes, comparing an
always-on gap junction against one gated off above threshold.

See [`README.md`](chapter21.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from ipywidgets import interact

## LIF Network with a Gap Junction

Two LIF neurons diffusively coupled by a gap junction, with an optional
extra $\epsilon$ kick to the other cell's voltage whenever a cell spikes
($\epsilon=0$ isolates the pure diffusive coupling from that
spike-triggered kick).

In [ ]:
def simulate_lif_network_with_gj(epsilon, tau=10.0, g_gap=0.01, i_ext=None,
                                  t_final=100.0, dt=0.002):
    num = 2
    if i_ext is None:
        i_ext = np.array([0.125, 0.09])
    G = np.array([[0.0, g_gap], [g_gap, 0.0]])
    c = G.sum(axis=0)
    m_steps = round(t_final / dt)

    v = np.zeros((num, m_steps + 1))
    v[:, 0] = [0.4, 0.9]
    spike_times = [[], []]

    for k in range(m_steps):
        v_inc = -v[:, k] / tau + i_ext + G @ v[:, k] - c * v[:, k]
        v[:, k + 1] = v[:, k] + dt * v_inc
        w = v[:, k + 1]
        ind = int(np.argmax(w))
        if w[ind] > 1:
            v[ind, k + 1] = 0.0
            other = 1 - ind
            v[other, k + 1] += epsilon
            if v[other, k + 1] > 1:
                v[other, k + 1] = 0.0
            spike_times[ind].append((k + 1) * dt)

    return v, spike_times


def plot_lif_network_with_gj(v_coupled, spikes_coupled, v_uncoupled, spikes_uncoupled,
                              t_final=100.0, dt=0.002):
    t = np.arange(v_coupled.shape[1]) * dt
    fig, ax = plt.subplots(2, figsize=(7, 6))

    for tt in spikes_coupled[0]:
        ax[0].plot([tt, tt], [0, 6], '-k', linewidth=3)
    for tt in spikes_uncoupled[0]:
        ax[0].plot([tt, tt], [0, 6], '-r', linewidth=1)
    ax[0].plot(t, v_coupled[0], '-k', linewidth=3)
    ax[0].plot(t, v_uncoupled[0], '-r', linewidth=1)
    ax[0].set_ylabel('$v_1$ [mV]')
    ax[0].set_xlim(0, t_final)
    ax[0].set_ylim(0, 6)

    for tt in spikes_coupled[1]:
        ax[1].plot([tt, tt], [0, 6], '-k', linewidth=3)
    for tt in spikes_uncoupled[1]:
        ax[1].plot([tt, tt], [0, 6], '-r', linewidth=1)
    ax[1].plot(t, v_coupled[1], '-k', linewidth=3)
    ax[1].plot(t, v_uncoupled[1], '-r', linewidth=1)
    ax[1].set_xlabel('$t$ [ms]')
    ax[1].set_ylabel('$v_2$ [mV]')
    ax[1].set_xlim(0, t_final)
    ax[1].set_ylim(0.85, 0.95)

    plt.tight_layout()
    plt.show()

In [ ]:
beta = 3.0
g_gap = 0.01
v_coupled, spikes_coupled = simulate_lif_network_with_gj(beta * g_gap)
v_uncoupled, spikes_uncoupled = simulate_lif_network_with_gj(0.0)
plot_lif_network_with_gj(v_coupled, spikes_coupled, v_uncoupled, spikes_uncoupled)

## WB Gating (shared by the `RESET_THRESHOLD`/`WB_NETWORK_WITH_GJ*` examples below)

In [ ]:
def alpha_h(v):
    return 0.35 * exp(-(v + 58) / 20)


def alpha_m(v):
    return 0.1 * (v + 35) / (1 - exp(-(v + 35) / 10))


def alpha_n(v):
    return 0.05 * (v + 34) / (1 - exp(-0.1 * (v + 34)))


def beta_h(v):
    return 5.0 / (exp(-0.1 * (v + 28)) + 1)


def beta_m(v):
    return 4 * exp(-(v + 60) / 18)


def beta_n(v):
    return 0.625 * exp(-(v + 44) / 80)


def m_inf(v):
    return alpha_m(v) / (alpha_m(v) + beta_m(v))


def h_inf(v):
    return alpha_h(v) / (alpha_h(v) + beta_h(v))


def n_inf(v):
    return alpha_n(v) / (alpha_n(v) + beta_n(v))

## WB Reset/Threshold Reference Trace

A single WB neuron's voltage trace, with two reference levels marked --
used to pick the reset/threshold pair for the LIF approximation used
above.

In [ ]:
def simulate_reset_threshold(c=1.0, g_k=9.0, g_na=35.0, g_l=0.1,
                              v_k=-90.0, v_na=55.0, v_l=-65.0,
                              i_ext=0.75, t_final=100.0, dt=0.01, v0=-63.0):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    m = np.zeros(m_steps + 1)
    h = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    v[0] = v0
    m[0] = m_inf(v[0])
    h[0] = h_inf(v[0])
    n[0] = n_inf(v[0])

    for k in range(m_steps):
        v_inc = (g_k * n[k] ** 4 * (v_k - v[k]) + g_na * m[k] ** 3 * h[k] * (v_na - v[k])
                 + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        h_inc = alpha_h(v[k]) * (1 - h[k]) - beta_h(v[k]) * h[k]

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc

    t = np.arange(m_steps + 1) * dt
    return t, v


def plot_reset_threshold(t, v, t_final=100.0):
    plt.figure(figsize=(7, 4))
    plt.plot(t, v, '-k', linewidth=2)
    plt.plot([0, t_final], [-67, -67], '--b', linewidth=1)
    plt.plot([0, t_final], [-52, -52], '--b', linewidth=1)
    plt.xlim(0, t_final)
    plt.ylim(-100, 50)
    plt.xlabel('$t$ [ms]')
    plt.ylabel('$v$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_reset_threshold(*simulate_reset_threshold())

## WB Network with a Gap Junction

Two WB neurons coupled by a gap junction, integrated with `odeint` --
neuron 1 (driven, $I=1.0$) and neuron 2 (silent, $I=0$) show how spiking
activity couples into a subthreshold neighbor.

In [ ]:
def _wb_network_derivative(x0, t, N, g_k, g_na, g_l, v_k, v_na, v_l, g_gap, i_ext):
    df = np.zeros(3 * N)
    v = x0[:N]
    h = x0[N:2 * N]
    n = x0[2 * N:]

    for i in range(N):
        m_i = alpha_m(v[i]) / (alpha_m(v[i]) + beta_m(v[i]))
        i_na = g_na * m_i ** 3 * h[i] * (v[i] - v_na)
        i_l = g_l * (v[i] - v_l)
        i_k = g_k * n[i] ** 4 * (v[i] - v_k)

        i_syn = 0.0
        for j in range(N):
            if i != j:
                i_syn += (v[j] - v[i])
        i_syn *= g_gap

        df[i] = -i_na - i_k - i_l + i_syn + i_ext[i]
        df[i + N] = alpha_h(v[i]) * (1 - h[i]) - beta_h(v[i]) * h[i]
        df[i + 2 * N] = alpha_n(v[i]) * (1 - n[i]) - beta_n(v[i]) * n[i]

    return df


def simulate_wb_network_with_gj(g_k=9.0, g_na=35.0, g_l=0.1, v_k=-90.0, v_na=55.0, v_l=-65.0,
                                 i_ext=None, g_gap=0.01, t_final=200.0, dt=0.01, v0=-63.0):
    N = 2
    if i_ext is None:
        i_ext = [1.0, 0.0]
    v = np.full(N, v0)
    h = h_inf(v)
    n = n_inf(v)
    x0 = v.tolist() + h.tolist() + n.tolist()

    t = np.arange(0, t_final, dt)
    sol = odeint(_wb_network_derivative, x0, t, args=(N, g_k, g_na, g_l, v_k, v_na, v_l, g_gap, i_ext))
    return t, sol[:, 0], sol[:, 1]


def plot_wb_network_with_gj(t, v1, v2):
    fig, ax = plt.subplots(2, figsize=(7, 5), sharex=True)
    ax[0].plot(t, v1, lw=2, c="k")
    ax[1].plot(t, v2, lw=2, c="k")

    for a in ax:
        a.set_xlim(100, max(t))
        a.set_xlabel("time [ms]")
    ax[0].set_yticks(range(-100, 100, 50))
    ax[0].set_ylabel("v1 [mV]")
    ax[1].set_ylabel("v2 [mV]")
    ax[0].set_ylim(-100, 50)
    ax[1].set_ylim(-64, -62)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_wb_network_with_gj(*simulate_wb_network_with_gj())

## WB Network with a Gap Junction: Subthreshold Coupling

Compares an always-on gap junction against one gated off above a voltage
threshold, isolating electrical equalization from the spiking dynamics
that follow it.

In [ ]:
def simulate_wb_network_with_gj_subthreshold(gap_gate, c=1.0, g_k=9.0, g_na=35.0, g_l=0.1,
                                              v_k=-90.0, v_na=55.0, v_l=-65.0, i_ext=None,
                                              t_final=100.0, dt=0.01, v0=-63.0):
    '''gap_gate(v1) returns the 2x2 coupling matrix as a function of neuron
    1's current voltage -- constant for the always-on gap junction, zeroed
    out above threshold for the subthreshold-only one.'''
    num = 2
    if i_ext is None:
        i_ext = np.array([1.0, 0.0])
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros((num, m_steps + 1))
    m = np.zeros((num, m_steps + 1))
    h = np.zeros((num, m_steps + 1))
    n = np.zeros((num, m_steps + 1))
    v[:, 0] = [v0, v0]
    m[:, 0] = m_inf(v[:, 0])
    h[:, 0] = h_inf(v[:, 0])
    n[:, 0] = n_inf(v[:, 0])

    for k in range(m_steps):
        g_gap = gap_gate(v[0, k])
        d_gap = g_gap.sum(axis=0)

        v_inc = (g_k * n[:, k] ** 4 * (v_k - v[:, k]) + g_na * m[:, k] ** 3 * h[:, k] * (v_na - v[:, k])
                 + g_l * (v_l - v[:, k]) + g_gap @ v[:, k] - d_gap * v[:, k] + i_ext) / c
        n_inc = alpha_n(v[:, k]) * (1 - n[:, k]) - beta_n(v[:, k]) * n[:, k]
        h_inc = alpha_h(v[:, k]) * (1 - h[:, k]) - beta_h(v[:, k]) * h[:, k]

        v_tmp = v[:, k] + dt05 * v_inc
        h_tmp = h[:, k] + dt05 * h_inc
        n_tmp = n[:, k] + dt05 * n_inc
        m_tmp = m_inf(v_tmp)

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + g_gap @ v_tmp - d_gap * v_tmp + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp

        v[:, k + 1] = v[:, k] + dt * v_inc
        h[:, k + 1] = h[:, k] + dt * h_inc
        n[:, k + 1] = n[:, k] + dt * n_inc
        m[:, k + 1] = m_inf(v[:, k + 1])

    return v


def plot_wb_network_with_gj_subthreshold(v_always, v_sub, t_final=100.0, dt=0.01):
    t = np.arange(v_always.shape[1]) * dt
    half = v_always.shape[1] // 2

    fig, ax = plt.subplots(2, figsize=(7, 6))

    ax[0].plot(t, v_always[0], '-k', linewidth=3)
    ax[0].plot(t, v_sub[0], '-r', linewidth=1)
    ax[0].set_ylabel('$v_1$ [mV]')
    ax[0].set_xlim(0, t_final)
    ax[0].set_ylim(-100, 50)

    ax[1].plot(t, v_always[1], '-k', linewidth=3)
    ax[1].plot(t, v_sub[1], '-r', linewidth=1)
    ax[1].set_xlabel('$t$ [ms]')
    ax[1].set_ylabel('$v_2$ [mV]')
    ax[1].set_xlim(0, t_final)
    tail = v_always[1, half:]
    ax[1].set_ylim(tail.min() - 0.75, tail.max() + 0.75)

    plt.tight_layout()
    plt.show()

In [ ]:
gap_strength = 0.01
gap_always_on = np.array([[0.0, gap_strength], [gap_strength, 0.0]])
v_always = simulate_wb_network_with_gj_subthreshold(lambda v1: gap_always_on)


def gap_subthreshold(v1, thr=-50.0):
    return gap_always_on if v1 <= thr else np.zeros((2, 2))


v_sub = simulate_wb_network_with_gj_subthreshold(gap_subthreshold)
plot_wb_network_with_gj_subthreshold(v_always, v_sub)